# Proyecto final · 00 · Configuración

Este notebook crea el catálogo, los esquemas y el volume donde aterrizan tus archivos.

**Antes de correr nada:** busca y reemplaza `apellido` por tu apellido, en minúsculas y sin tildes, en **los seis notebooks** de la carpeta. Busca otra vez para confirmar que no quedó ninguno.

La carpeta trae un **dataset de ejemplo** (las ventas de `samples.bakehouse`) para que corra completa desde el primer día. Tu entrega usa **tus propios datos**: cuando los tengas, súbelos al volume `landing` y cambia la ruta y el formato en `01_bronze`.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS pf_fallas;

CREATE SCHEMA IF NOT EXISTS pf_fallas.bronze;
CREATE SCHEMA IF NOT EXISTS pf_fallas.silver;
CREATE SCHEMA IF NOT EXISTS pf_fallas.gold;

CREATE VOLUME IF NOT EXISTS pf_fallas.bronze.landing;

## Datos de ejemplo, en lotes

Cada corrida de esta celda escribe **un lote** de archivos JSON en `landing/ventas/lote_N`. Sirve para probar que tu ingesta es incremental: corre `01_bronze`, cambia el lote a 2, vuelve a correr esta celda y después `01_bronze`. La segunda vez solo debe entrar el lote nuevo.

Si ya trabajas con tus datos, no necesitas esta celda.

In [0]:
# Crear las carpetas para los dos lotes
dbutils.fs.mkdirs("/Volumes/pf_fallas/bronze/landing/vehiculos/lote_1")
dbutils.fs.mkdirs("/Volumes/pf_fallas/bronze/landing/vehiculos/lote_2")

# Verificar las carpetas
display(dbutils.fs.ls("/Volumes/pf_fallas/bronze/landing/vehiculos/"))

path,name,size,modificationTime
dbfs:/Volumes/pf_fallas/bronze/landing/vehiculos/lote_1/,lote_1/,0,1792015986233
dbfs:/Volumes/pf_fallas/bronze/landing/vehiculos/lote_2/,lote_2/,0,1792015986233


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/pf_fallas/bronze/landing/vehiculos/lote_1/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/pf_fallas/bronze/landing/vehiculos/lote_1/ev_lote_1.csv,ev_lote_1.csv,3538872,1792016283000


### C2 · Orquestación

El Job `pf_fallas_ev` ejecuta las tareas `bronze`, `silver` y `gold`
en ese orden. Cada tarea depende del éxito de la anterior.

La tarea `bronze` tiene dos reintentos para atender fallos temporales.
El Job cuenta con una programación diaria.

La ejecución completa terminó correctamente. La prueba que inserta una
cantidad negativa se conserva en `03_gold` como evidencia de calidad,
pero se excluye de `03_gold_job` para permitir la ejecución automática.